In [ ]:
# Cell 0: Setup
import os, json, subprocess, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
WORK = '/content/eval221'
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{WORK}/audio', exist_ok=True)
os.makedirs(f'{WORK}/labels', exist_ok=True)

eval_vids = ['LWYfo_8t5WQ','oiRyNnyG698','KVMbGry8AgM','xvNoe0HZnVc',
              'QPywJakXcc0','iiyLmS-0GFQ','pbYlQ-fZHWk','MMpTINFg9dQ',
              'JLqTpOqhGXo','tRN6rYyr9Bk','J1NYJu9ENMg','OMxKP1eBR_A',
              '6JQzl2LlXbQ','l2oaxKORheA','_QdTi-N_Pgk','1tO9MWWOgHk']
eval_vids = sorted(set(eval_vids))
print(f'Eval videos: {len(eval_vids)}')


In [ ]:
# Cell 1: Download audio via yt-dlp
def dl_yt(vid):
    out = f'{WORK}/audio/{vid}.wav'
    if os.path.exists(out): return True
    cmd = ['yt-dlp', '-f', 'bestaudio[ext=m4a]', '-o', f'{out}.%(ext)s',
           f'https://www.youtube.com/watch?v={vid}',
           '--no-playlist', '--quiet', '--socket-timeout', '90',
           '--extract-audio', '--audio-format', 'wav']
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    if r.returncode == 0:
        for ext in ['m4a','webm','mp4']:
            tmp = f'{out}.{ext}'
            if os.path.exists(tmp):
                os.rename(tmp, out)
        return os.path.exists(out)
    return False

ok, fail = 0, 0
for i, vid in enumerate(eval_vids):
    ok2 = dl_yt(vid)
    print(f'  [{i+1}/{len(eval_vids)}] {vid} {"OK" if ok2 else "FAIL"}')
    if ok2: ok += 1
    else: fail += 1

n = len([f for f in os.listdir(f'{WORK}/audio') if f.endswith('.wav')])
print(f'Audio: {n}/{len(eval_vids)}')


In [ ]:
# Cell 2: Mount Drive for labels
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
BASE = '/content/drive/MyDrive/standup4ai'

import shutil

LABEL_SRC = f'{BASE}/labels'
if os.path.exists(LABEL_SRC):
    for vid in eval_vids:
        src = f'{LABEL_SRC}/{vid}.csv'
        dst = f'{WORK}/labels/{vid}.csv'
        if os.path.exists(src) and not os.path.exists(dst):
            shutil.copy(src, dst)
    n_labels = len([f for f in os.listdir(f'{WORK}/labels') if f.endswith('.csv')])
    print(f'Labels: {n_labels}')
else:
    print(f'Labels not at {LABEL_SRC}')


In [ ]:
# Cell 3: Load models + features
import torch, torch.nn as nn
from transformers import AutoModel
import numpy as np, librosa
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device); wavlm.eval()
print('WavLM ready')

class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1), nn.Sigmoid())
    def forward(self, x): return self.net(x)

MODEL_PATH = f'{BASE}/scale221/scale221_fusion_model.pt'
if not os.path.exists(MODEL_PATH):
    MODEL_PATH = f'{BASE}/experiments/scale221_fusion_model.pt'
model = FusionMLP()
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'), strict=False)
model.eval()
print(f'scale221 loaded')

SR_WAVLM, SR_PROSODY = 16000, 22050

def prosody23(y, sr):
    f = []
    try:
        f0, vd, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]; v = vd[~np.isnan(f0)]
        f.extend([np.mean(f0c) if len(f0c)>0 else 0,
                  np.std(f0c) if len(f0c)>0 else 0,
                  np.max(f0c) if len(f0c)>0 else 0,
                  np.min(f0c) if len(f0c)>0 else 0,
                  np.mean(v) if len(v)>0 else 0])
    except: f.extend([0]*5)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    dur = len(y)/sr
    f.extend([dur, dur/(np.sum(rms>np.mean(rms))+1])
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except: f.extend([0]*5)
    try:
        yh, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except: f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def word_feat(y16, y22, t0, t1):
    if t1-t0 < 0.05: return None
    s16, e16 = int(t0*SR_WAVLM), min(int(t1*SR_WAVLM), len(y16))
    c16 = y16[s16:e16]
    if len(c16) < 0.5*SR_WAVLM: return None
    if len(c16) < 5*SR_WAVLM: c16 = np.pad(c16, (0, int(5*SR_WAVLM)-len(c16)))
    with torch.no_grad():
        wemb = wavlm(torch.tensor(c16/32768.0).unsqueeze(0).to(device)).last_hidden_state.mean(1).squeeze().cpu().numpy()
    s22, e22 = int(t0*SR_PROSODY), min(int(t1*SR_PROSODY), len(y22))
    pros = prosody23(y22[s22:e22], SR_PROSODY)
    return np.concatenate([wemb, pros])

print('Ready')


In [ ]:
# Cell 4: IoU metrics
import pandas as pd

def bio2spans(df):
    spans, i = [], 0
    while i < len(df):
        lbl = str(df.iloc[i].get('label','')).strip()
        ts = eval(str(df.iloc[i]['timestamp']))
        if lbl == 'L': spans.append((float(ts[0]), float(ts[1])))
        elif lbl == 'B':
            st, en = float(ts[0]), float(ts[1])
            j = i+1
            while j < len(df):
                nl = str(df.iloc[j].get('label','')).strip()
                if nl in ('I','L'): en = float(eval(str(df.iloc[j]['timestamp']))[1]); j+=1
                else: break
            spans.append((st, en)); i = j-1
        i += 1
    return spans

def iou(s1, s2):
    inter = max(0.0, min(s1[1],s2[1])-max(s1[0],s2[0]))
    union = max(s1[1],s2[1])-min(s1[0],s2[0])
    return inter/union if union > 0 else 0.0

def segf1(pred, gt, th=0.3):
    if not pred or not gt: return 0.0, 0.0, 0.0
    mp, mg = set(), set()
    for pi, ps in enumerate(pred):
        bi, bg = 0.0, -1
        for gi, gs in enumerate(gt):
            if gi in mg: continue
            iv = iou(ps, gs)
            if iv >= th and iv > bi: bi, bg = iv, gi
        if bg >= 0: mp.add(pi); mg.add(bg)
    tp = len(mp)
    p = tp/len(pred) if pred else 0.0
    r = tp/len(gt) if gt else 0.0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return p, r, f

def merge(probs, ts, thr=0.5):
    spans, in_seg, start = [], False, 0.0
    for i, (pr, (t0, t1)) in enumerate(zip(probs, ts)):
        if pr >= thr and not in_seg: in_seg, start = True, t0
        elif pr < thr and in_seg: in_seg = False; spans.append((start, t0))
    if in_seg: spans.append((start, ts[-1][1]))
    return spans

print('Metrics ready')


In [ ]:
# Cell 5: Evaluate
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]
RESULTS = {th: [] for th in THRESHOLDS}
per_video = []

for vid in tqdm(eval_vids):
    ap = None
    for ext in ['.wav', '.m4a']:
        p = f'{WORK}/audio/{vid}{ext}'
        if os.path.exists(p): ap = p; break
    if not ap: print(f'No audio: {vid}'); continue
    lp = f'{WORK}/labels/{vid}.csv'
    if not os.path.exists(lp): print(f'No labels: {vid}'); continue
    try: df = pd.read_csv(lp)
    except: continue
    ts = []
    for _, row in df.iterrows():
        try: ts.append((float(eval(str(row['timestamp']))[0], float(eval(str(row['timestamp']))[1]))
        except: pass
    try:
        y22, _ = librosa.load(ap, sr=SR_PROSODY, mono=True)
        y16, _ = librosa.load(ap, sr=SR_WAVLM, mono=True)
    except: continue
    feats, mask = [], []
    for t0, t1 in ts:
        f = word_feat(y16, y22, t0, t1)
        if f is None:
            feats.append(np.zeros(791, dtype=np.float32)); mask.append(False)
        else:
            feats.append(f); mask.append(True)
    with torch.no_grad():
        probs = model(torch.tensor(np.array(feats), dtype=torch.float32)).numpy().squeeze()
    probs = np.where(mask, probs, 0.0)
    gt = bio2spans(df)
    if not gt: continue
    pred = merge(probs, ts, 0.5)
    row = {'vid': vid, 'n_pred': len(pred), 'n_gt': len(gt)}
    for th in THRESHOLDS:
        p, r, f = segf1(pred, gt, th)
        row[f'p_{th}'] = round(p, 4)
        row[f'r_{th}'] = round(r, 4)
        row[f'f_{th}'] = round(f, 4)
        RESULTS[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})
    per_video.append(row)

print(f'Evaluated: {len(per_video)}/{len(eval_vids)}')


In [ ]:
# Cell 6: Results
print('='*60)
print('SCALE221 EXTERNAL EVALUATION')
print(f'N = {len(per_video)} videos')
print('='*60)
print(f"{'IoU':>6} | {'P':>10} {'R':>10} {'F1':>10}")
print('-'*45)
for th in THRESHOLDS:
    rs = RESULTS[th]
    if not rs: continue
    pm = np.mean([x['p'] for x in rs])
    rm = np.mean([x['r'] for x in rs])
    fm = np.mean([x['f'] for x in rs])
    print(f' >={th:.1f} | {pm:.4f} {rm:.4f} {fm:.4f}')
print()
for row in sorted(per_video, key=lambda x: x.get('f_0.3', 0), reverse=True)[:8]:
    print(f"  {row['vid']:<18} gt={row['n_gt']:>3} pd={row['n_pred']:>4} F1={row.get('f_0.3', 0):.4f}")

out = {'n_videos': len(per_video), 'per_video': per_video}
with open(f'{WORK}/results.json', 'w') as f:
    json.dump(out, f, indent=2)
print(f'Saved: {WORK}/results.json')
